In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from mal_client import MALClient
from anime_data import AnimeDataClient
from anime_recommender import SimilarityRecommender

load_dotenv(PROJECT_ROOT / ".env")

client_id = os.getenv("CLIENT_ID")

In [3]:
anime_data_client = AnimeDataClient(client_id, cache_file=PROJECT_ROOT / "anime_cache.json")

In [4]:
anime_data = anime_data_client.get_cache()

Finding best numerical feature set

In [5]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

numeric_candidates = pd.DataFrame(anime_data.values())

statistics_df = pd.json_normalize(numeric_candidates["statistics"])
statistics_df = statistics_df.rename(columns={
    "num_list_users": "statistics_num_list_users",
    "status.watching": "watching",
    "status.completed": "completed",
    "status.on_hold": "on_hold",
    "status.dropped": "dropped",
    "status.plan_to_watch": "plan_to_watch",
})
statistics_df = statistics_df.apply(pd.to_numeric, errors="coerce")

numeric_candidates = pd.concat(
    [numeric_candidates.drop(columns=["statistics"]), statistics_df],
    axis=1,
)

numeric_columns = [
    "mean",
    "rank",
    "popularity",
    "num_list_users",
    "num_scoring_users",
    "num_episodes",
    "statistics_num_list_users",
    "watching",
    "completed",
    "on_hold",
    "dropped",
    "plan_to_watch",
]

numeric_analysis_df = (
    numeric_candidates[numeric_columns]
    .apply(pd.to_numeric, errors="coerce")
    .dropna()
)

numeric_analysis_df.head()

,mean,rank,popularity,num_list_users,num_scoring_users,num_episodes,statistics_num_list_users,watching,completed,on_hold,dropped,plan_to_watch
0,8.89,26.0,118,1314117,505249,74,1313880,177147,501891,94398,43705,496739
1,8.12,549.0,1195,234811,80450,26,234759,14043,94627,9711,6501,109877
2,9.11,3.0,3,3659053,2298157,64,3658804,285524,2645506,120875,64923,541976
3,9.03,10.0,8,3164257,1965469,148,3164039,381795,2174185,151790,69066,387203
4,8.24,385.0,270,819854,289082,25,819715,56032,322940,37493,36737,366513


In [ ]:
def calculate_vif(df):
    rows = []
    for target_col in df.columns:
        X = df.drop(columns=[target_col]).to_numpy(dtype=float)
        y = df[target_col].to_numpy(dtype=float)

        model = LinearRegression()
        model.fit(X, y)
        r_squared = model.score(X, y)

        vif = np.inf if np.isclose(1 - r_squared, 0) else 1 / (1 - r_squared)
        rows.append({
            "feature": target_col,
            "r_squared_from_other_features": r_squared,
            "vif": vif,
        })

    return pd.DataFrame(rows).sort_values("vif", ascending=False)

reduced_numeric_analysis_df = numeric_analysis_df.drop(
    columns=[
        "completed",
        "on_hold",
        "statistics_num_list_users",
        "watching",
        "dropped",
        "plan_to_watch",
        "num_list_users",
        "num_scoring_users",
        # "num_episodes",
        "rank",
        # "popularity",
        # "mean",
    ],
    errors="ignore",
)

vif_results = calculate_vif(reduced_numeric_analysis_df)
vif_results

,feature,r_squared_from_other_features,vif
1,popularity,0.257899,1.347526
3,on_hold,0.246880,1.327809
0,mean,0.150187,1.176730
2,num_episodes,0.079698,1.086600


Build features

In [22]:
from anime_features import AnimeFeatureBuilder

builder = AnimeFeatureBuilder(
    anime_data,
    max_tfidf_features=3000,
    n_svd_components=300
)

anime_df_num = builder.build_num_features().set_index("id")
anime_genres_df = builder.build_genre_features().set_index("anime_id")
anime_studios_df = builder.build_studio_features().set_index("anime_id")
synopsis_tfidf, synopsis_features = builder.build_synopsis_features()
synopsis_svd_df = builder.apply_svd(synopsis_tfidf, synopsis_features.index)

anime_df_complete = pd.concat([
    anime_df_num,
    anime_genres_df,
    synopsis_svd_df,
], axis=1).dropna()

builder.svd_explained_variance

np.float64(0.41006426159563164)

In [63]:
anime_df_num_new = pd.DataFrame(anime_data.values())
anime_df_num_new = anime_df_num_new.drop(
    columns=[
        "main_picture",
        "title",
        "synopsis",
        "media_type",
        "status",
        "genres",
        "rating",
        "recommendations",
        "studios",
    ],
    errors="ignore",
)

statistics_df = pd.json_normalize(anime_df_num_new["statistics"])
statistics_df = statistics_df.rename(columns={
    "num_list_users": "statistics_num_list_users",
    "status.watching": "watching",
    "status.completed": "completed",
    "status.on_hold": "on_hold",
    "status.dropped": "dropped",
    "status.plan_to_watch": "plan_to_watch",
})
statistics_df = statistics_df.apply(pd.to_numeric, errors="coerce")

anime_df_num_new = pd.concat([anime_df_num_new.drop(columns=["statistics"]), statistics_df], axis=1)

anime_df_num_new = anime_df_num_new.drop(
    columns=[
        "completed",
        "on_hold",
        "statistics_num_list_users",
        # "watching",
        "dropped",
        "plan_to_watch",
        "num_list_users",
        "num_scoring_users",
        "num_episodes",
        "rank",
        # "popularity",
        # "mean",
    ],
    errors="ignore",).set_index("id").dropna()

anime_df_num_new

,mean,popularity,watching
id,,,
19,8.89,118,177147
1827,8.12,1195,14043
5114,9.11,3,285524
11061,9.03,8,381795
13125,8.24,270,56032
...,...,...,...
16512,6.83,1602,10897
873,6.86,2826,2884
32032,6.71,2230,9373


In [64]:
# Pick the feature set to use below.
# anime_df = anime_df_complete
anime_df = pd.concat([anime_df_num_new, anime_genres_df, synopsis_svd_df], axis=1).dropna()
# anime_df = pd.concat([anime_df_complete, anime_studios_df], axis=1).dropna()
# anime_df = anime_df_num.dropna()
# anime_df = anime_genres_df.dropna()
# anime_df = anime_studios_df.dropna()
# anime_df = pd.concat([anime_df_num, anime_genres_df], axis=1).dropna()
# anime_df = pd.concat([anime_df_num, anime_studios_df], axis=1).dropna()
# anime_df = pd.concat([anime_df_num, synopsis_svd_df], axis=1).dropna()
# anime_df = pd.concat([anime_genres_df, synopsis_svd_df], axis=1).dropna()
# anime_df = pd.concat([anime_genres_df, anime_studios_df], axis=1).dropna()
# anime_df = pd.concat([anime_df_num, anime_genres_df, anime_studios_df], axis=1).dropna()
# anime_df = pd.concat([anime_df_num, synopsis_svd_df, anime_studios_df], axis=1).dropna()

Convert each anime in df to vectors

In [65]:
recommender = SimilarityRecommender()
anime_vectors = recommender.create_anime_vectors(anime_df)
anime_df_scaled = recommender.anime_df_scaled

anime_df_scaled.head()

,mean,popularity,watching,Action,Adult Cast,Adventure,Anthropomorphic,Avant Garde,Award Winning,Boys Love,...,synopsis_svd_290,synopsis_svd_291,synopsis_svd_292,synopsis_svd_293,synopsis_svd_294,synopsis_svd_295,synopsis_svd_296,synopsis_svd_297,synopsis_svd_298,synopsis_svd_299
19,2.443208,-1.038643,3.221810,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,-0.019556,0.005819,0.054397,-0.039712,0.102000,0.006899,-0.045972,0.024398,0.021041,0.052458
1827,1.236685,-0.652717,-0.141936,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,-0.005089,0.014942,-0.050817,-0.031403,0.010778,-0.002184,-0.005891,0.062198,0.004281,-0.020533
5114,2.787928,-1.079851,5.456903,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.008645,-0.046776,-0.047862,-0.018033,-0.022562,-0.014848,0.000583,-0.000781,-0.030836,-0.045200
11061,2.662576,-1.078059,7.442331,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,-0.010859,0.026910,0.043255,-0.045991,-0.046760,-0.006471,0.018076,0.048248,-0.010416,0.027548
13125,1.424715,-0.984176,0.724017,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.009030,-0.020140,0.030945,0.034260,0.003442,-0.038289,0.031925,-0.018523,0.016452,-0.016687


Get user Data

In [66]:
username = "chekkit"
user_client = MALClient(client_id)

user_data = user_client.get_user_data(username)
user_scores = user_client.get_scores(user_data)

Hit Rate

In [67]:
from anime_evaluation import HitRateEvaluator

n_runs = 1000
result_top_ks = (5, 10)
uncertainty_weight = 7.5

evaluator = HitRateEvaluator(
    anime_df_scaled=anime_df_scaled,
    anime_df=anime_df,
    scores=user_scores,
    heldout_fraction=0.25,
)

(
    bayesian_results,
    bayesian_summary,
    best_bayesian_weights,
    baseline_results,
    baseline_summary,
) = evaluator.tune_bayesian_uncertainty(
    weights=[uncertainty_weight],
    n_runs=n_runs,
    top_ks=result_top_ks,
    random_state=42,
)

if baseline_summary is not None and not baseline_summary.empty:
    average_metrics = bayesian_summary.merge(
        baseline_summary,
        on="k",
        how="left",
    )
else:
    average_metrics = bayesian_summary.copy()

average_metrics = average_metrics.rename(
    columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
)

average_metrics

,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits
0,7.5,5,0.643,0.220907,0.103710,0.035630,3.215,0.0984,0.122689,0.015871,0.019788,0.492
1,7.5,10,0.402,0.145661,0.129677,0.046988,4.020,0.0643,0.059658,0.020742,0.019244,0.643


## Numeric Combo Sweep

This sweep keeps the core numeric signals (`mean`, `popularity`, `num_episodes`) and tests all 64 combinations of optional activity/support signals inside the full feature context: numeric combo + genres + synopsis SVD. Use a smaller run count for the sweep, then rerun the top few combos with more runs.

In [14]:
from itertools import combinations
from anime_evaluation import HitRateEvaluator
from anime_recommender import SimilarityRecommender

core_numeric_features = ["mean", "popularity", "num_episodes"]
optional_numeric_features = [
    "num_scoring_users",
    "watching",
    "completed",
    "on_hold",
    "dropped",
    "plan_to_watch",
]

numeric_combo_base = numeric_candidates[["id"] + numeric_columns].copy()
for column in numeric_columns:
    numeric_combo_base[column] = pd.to_numeric(
        numeric_combo_base[column],
        errors="coerce",
    )
numeric_combo_base = numeric_combo_base.set_index("id").dropna()

combo_n_runs = 20
combo_top_ks = (5, 10)
combo_uncertainty_weight = 7.5
combo_rows = []

for combo_size in range(len(optional_numeric_features) + 1):
    for optional_combo in combinations(optional_numeric_features, combo_size):
        selected_numeric_features = core_numeric_features + list(optional_combo)
        feature_set_name = "core"
        if optional_combo:
            feature_set_name += "_" + "_".join(optional_combo)

        combo_anime_df = pd.concat(
            [
                numeric_combo_base[selected_numeric_features],
                anime_genres_df,
                synopsis_svd_df,
            ],
            axis=1,
        ).dropna()

        combo_recommender = SimilarityRecommender()
        combo_recommender.create_anime_vectors(combo_anime_df)

        combo_evaluator = HitRateEvaluator(
            anime_df_scaled=combo_recommender.anime_df_scaled,
            anime_df=combo_anime_df,
            scores=user_scores,
            heldout_fraction=0.25,
        )

        _, combo_summary, _, _, combo_baseline_summary = (
            combo_evaluator.tune_bayesian_uncertainty(
                weights=[combo_uncertainty_weight],
                n_runs=combo_n_runs,
                top_ks=combo_top_ks,
                random_state=42,
            )
        )

        if combo_baseline_summary is not None and not combo_baseline_summary.empty:
            combo_summary = combo_summary.merge(
                combo_baseline_summary,
                on="k",
                how="left",
            )

        for row in combo_summary.to_dict("records"):
            row["feature_set"] = feature_set_name
            row["numeric_features"] = ", ".join(selected_numeric_features)
            row["n_numeric_features"] = len(selected_numeric_features)
            row["n_runs"] = combo_n_runs
            combo_rows.append(row)

numeric_combo_results = pd.DataFrame(combo_rows)
numeric_combo_summary = numeric_combo_results.sort_values(
    ["k", "avg_precision_at_k", "avg_hit_rate"],
    ascending=[True, False, False],
)

numeric_combo_summary.head(20)


,uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits,feature_set,numeric_features,n_numeric_features,n_runs
4,7.5,5,0.64,0.178885,0.103226,0.028852,3.20,0.17,0.134164,0.027419,0.021639,0.85,core_watching,"mean, popularity, num_episodes, watching",4,20
10,7.5,5,0.63,0.217885,0.101613,0.035143,3.15,0.17,0.134164,0.027419,0.021639,0.85,core_dropped,"mean, popularity, num_episodes, dropped",4,20
8,7.5,5,0.62,0.214231,0.100000,0.034553,3.10,0.17,0.134164,0.027419,0.021639,0.85,core_on_hold,"mean, popularity, num_episodes, on_hold",4,20
18,7.5,5,0.60,0.215211,0.096774,0.034711,3.00,0.17,0.134164,0.027419,0.021639,0.85,core_num_scoring_users_on_hold,"mean, popularity, num_episodes, num_scoring_us...",5,20
34,7.5,5,0.60,0.224781,0.096774,0.036255,3.00,0.17,0.134164,0.027419,0.021639,0.85,core_completed_dropped,"mean, popularity, num_episodes, completed, dro...",5,20
38,7.5,5,0.60,0.215211,0.096774,0.034711,3.00,0.17,0.134164,0.027419,0.021639,0.85,core_on_hold_dropped,"mean, popularity, num_episodes, on_hold, dropped",5,20
20,7.5,5,0.59,0.219809,0.095161,0.035453,2.95,0.17,0.134164,0.027419,0.021639,0.85,core_num_scoring_users_dropped,"mean, popularity, num_episodes, num_scoring_us...",5,20
32,7.5,5,0.59,0.219809,0.095161,0.035453,2.95,0.17,0.134164,0.027419,0.021639,0.85,core_completed_on_hold,"mean, popularity, num_episodes, completed, on_...",5,20
42,7.5,5,0.59,0.210013,0.095161,0.033873,2.95,0.17,0.134164,0.027419,0.021639,0.85,core_dropped_plan_to_watch,"mean, popularity, num_episodes, dropped, plan_...",5,20
52,7.5,5,0.59,0.219809,0.095161,0.035453,2.95,0.17,0.134164,0.027419,0.021639,0.85,core_num_scoring_users_completed_on_hold,"mean, popularity, num_episodes, num_scoring_us...",6,20


## 255 Numeric Combo Sweep

This broader sweep tests all non-empty combinations of the reduced numeric candidate set inside the full feature context: numeric combo + genres + synopsis SVD. With 8 candidate numeric features, this produces `2^8 - 1 = 255` feature sets.

In [56]:
from itertools import combinations
from anime_evaluation import HitRateEvaluator
from anime_recommender import SimilarityRecommender

numeric_sweep_features = [
    "mean",
    "popularity",
    "num_scoring_users",
    "num_episodes",
    "watching",
    "completed",
    "dropped",
    "plan_to_watch",
]

numeric_combo_base = numeric_candidates[["id"] + numeric_sweep_features].copy()
for column in numeric_sweep_features:
    numeric_combo_base[column] = pd.to_numeric(
        numeric_combo_base[column],
        errors="coerce",
    )
numeric_combo_base = numeric_combo_base.set_index("id").dropna()

all_combo_n_runs = 20
all_combo_top_ks = (5, 10)
all_combo_uncertainty_weight = 7.5
all_combo_rows = []

for combo_size in range(1, len(numeric_sweep_features) + 1):
    for numeric_combo in combinations(numeric_sweep_features, combo_size):
        selected_numeric_features = list(numeric_combo)
        feature_set_name = "combo_" + "_".join(selected_numeric_features)

        combo_anime_df = pd.concat(
            [
                numeric_combo_base[selected_numeric_features],
                anime_genres_df,
                synopsis_svd_df,
            ],
            axis=1,
        ).dropna()

        combo_recommender = SimilarityRecommender()
        combo_recommender.create_anime_vectors(combo_anime_df)

        combo_evaluator = HitRateEvaluator(
            anime_df_scaled=combo_recommender.anime_df_scaled,
            anime_df=combo_anime_df,
            scores=user_scores,
            heldout_fraction=0.25,
        )

        _, combo_summary, _, _, combo_baseline_summary = (
            combo_evaluator.tune_bayesian_uncertainty(
                weights=[all_combo_uncertainty_weight],
                n_runs=all_combo_n_runs,
                top_ks=all_combo_top_ks,
                random_state=42,
            )
        )

        if combo_baseline_summary is not None and not combo_baseline_summary.empty:
            combo_summary = combo_summary.merge(
                combo_baseline_summary,
                on="k",
                how="left",
            )

        for row in combo_summary.to_dict("records"):
            row["feature_set"] = feature_set_name
            row["numeric_features"] = ", ".join(selected_numeric_features)
            row["n_numeric_features"] = len(selected_numeric_features)
            row["n_runs"] = all_combo_n_runs
            all_combo_rows.append(row)

all_numeric_combo_results = pd.DataFrame(all_combo_rows)
all_numeric_combo_summary = all_numeric_combo_results.sort_values(
    ["k", "avg_precision_at_k", "avg_hit_rate"],
    ascending=[True, False, False],
)

all_numeric_combo_summary.head(20)


,uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits,feature_set,numeric_features,n_numeric_features,n_runs
94,7.5,5,0.67,0.207998,0.108065,0.033548,3.35,0.17,0.134164,0.027419,0.021639,0.85,combo_mean_num_episodes_watching,"mean, num_episodes, watching",3,20
76,7.5,5,0.66,0.184676,0.106452,0.029786,3.30,0.17,0.134164,0.027419,0.021639,0.85,combo_mean_popularity_watching,"mean, popularity, watching",3,20
194,7.5,5,0.64,0.178885,0.103226,0.028852,3.20,0.17,0.134164,0.027419,0.021639,0.85,combo_mean_popularity_num_episodes_watching,"mean, popularity, num_episodes, watching",4,20
198,7.5,5,0.63,0.217885,0.101613,0.035143,3.15,0.17,0.134164,0.027419,0.021639,0.85,combo_mean_popularity_num_episodes_dropped,"mean, popularity, num_episodes, dropped",4,20
204,7.5,5,0.60,0.171679,0.096774,0.027690,3.00,0.17,0.134164,0.027419,0.021639,0.85,combo_mean_popularity_watching_dropped,"mean, popularity, watching, dropped",4,20
350,7.5,5,0.60,0.224781,0.096774,0.036255,3.00,0.17,0.134164,0.027419,0.021639,0.85,combo_mean_popularity_num_episodes_completed_d...,"mean, popularity, num_episodes, completed, dro...",5,20
328,7.5,5,0.59,0.219809,0.095161,0.035453,2.95,0.17,0.134164,0.027419,0.021639,0.85,combo_mean_popularity_num_scoring_users_num_ep...,"mean, popularity, num_scoring_users, num_episo...",5,20
354,7.5,5,0.59,0.210013,0.095161,0.033873,2.95,0.17,0.134164,0.027419,0.021639,0.85,combo_mean_popularity_num_episodes_dropped_pla...,"mean, popularity, num_episodes, dropped, plan_...",5,20
442,7.5,5,0.59,0.219809,0.095161,0.035453,2.95,0.17,0.134164,0.027419,0.021639,0.85,combo_mean_popularity_num_scoring_users_num_ep...,"mean, popularity, num_scoring_users, num_episo...",6,20
98,7.5,5,0.59,0.165116,0.095161,0.026632,2.95,0.17,0.134164,0.027419,0.021639,0.85,combo_mean_num_episodes_dropped,"mean, num_episodes, dropped",3,20


## Results

The strongest confirmed top-5 feature set was **combo_mean_popularity_watching_1000_runs**, which uses `mean`, `popularity`, and `watching` with genre one-hot features and synopsis SVD features. The best top-10 result still came from **core_watching_1000_runs**, which keeps `num_episodes` in the numeric set. The final checks below use **1000 repeated holdout runs** with Bayesian Ridge and an uncertainty weight of **7.5**.

| Feature set | P@5 | P@10 | Improvement over baseline |
| --- | ---: | ---: | ---: |
| combo_mean_popularity_watching_1000_runs | 0.643 | 0.402 | 6.53x / 6.25x |
| core_watching_1000_runs | 0.6282 | 0.4143 | 6.38x / 6.44x |
| combo_mean_num_episodes_watching_1000_runs | 0.6242 | 0.4129 | 6.34x / 6.42x |
| core_dropped_1000_runs | 0.6244 | 0.3812 | 6.35x / 5.93x |
| core_watching | 0.618 | 0.411 | 5.62x / 5.96x |
| core_dropped | 0.613 | 0.367 | 5.57x / 5.32x |
| all_features | 0.528 | 0.314 | 4.55x / 4.13x |
| activity_numeric_genres_svd | 0.244 | 0.210 | 2.10x / 2.76x |

The selected-feature results above were saved in `metrics/features_set_selection_20260615_153918.csv`. The 255-combo sweep was useful for shortlisting numeric feature sets, but the 1000-run checks are the better evidence because the holdout split variance is high. Studio features and the VIF-reduced activity numeric set did not improve recommendation quality.

Conclusion: use **mean + popularity + watching + genres + synopsis SVD** when optimizing for P@5. If P@10 matters more, **mean + popularity + num_episodes + watching + genres + synopsis SVD** is slightly stronger.